In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Veri setinin yüklenmesi
try:
    df = pd.read_csv(
    "Police_Transparency_-_Arrests_-_All_Data_(main_table___denormalized).csv"
)
    print("✔ Veri seti başarıyla yüklendi.")
except Exception as e:
    print(f"❌ Veri yükleme hatası: {e}")
    raise SystemExit

print(f"Veri boyutu: {df.shape}")
display(df.head())


✔ Veri seti başarıyla yüklendi.
Veri boyutu: (47444, 40)


,X,Y,rin,primary_key,arrest_type,arrest_translation,arrest_dt,arrest_time,arrest_hour_of_day,location,...,severity_code,severity_trans,ChargeClassTranslation,ChargeGrouping,arrestee_sex,arrestee_race,arrestee_ethnicity,arrestee_genderTranslation,arrestee_RaceAndEthnicity,ESRI_OID
0,693985,882475,110528,TE20226631,O,On-View: Arrested when first observed/investi...,2022/09/24 21:17:00+00,2117,21,2XX E 5TH ST,...,NaN,Unknown,Arizona Revised Statutes,Miscellaneous,M,B,N,Male,Black or African American,3715748
1,710663,880237,110739,TE20226842,T,Taken Into Custody: Arrest on warrant or PC f...,2022/10/04 17:30:00+00,1730,17,9XX S ACORN AVE,...,F,Felony,Arizona Revised Statutes,Drug charges,M,W,N,Male,White,3715749
2,704259,878441,110587,TE20226690,T,Taken Into Custody: Arrest on warrant or PC f...,2022/09/27 21:52:00+00,2152,21,1XXX E APACHE BLVD,...,M,Misdemeanor,Arizona Revised Statutes,Assault & related charges,M,W,H,Male,Hispanic or Latino,3715750
3,693160,868321,110521,TE20226624,O,On-View: Arrested when first observed/investi...,2022/09/24 21:00:00+00,2100,21,4XXX S MILL AVE,...,F,Felony,Arizona Revised Statutes,Miscellaneous,M,B,N,Male,Black or African American,3715751
4,708569,883842,110789,TE20226892,T,Taken Into Custody: Arrest on warrant or PC f...,2022/10/07 03:54:00+00,354,3,2XXX W RIO SALADO PKWY,...,M,Misdemeanor,Arizona Revised Statutes,Escape & related charges,M,W,H,Male,Hispanic or Latino,3715752


In [3]:
# Gereksiz sütunları kaldırma
columns_to_drop = [
    'X','Y','rin','primary_key','charge_rin','pin','ESRI_OID',
    'x_coordinate','y_coordinate',
    'arrest_officer',
    'ofc_age_range','ofc_genderTranslation','ofc_RaceAndEthnicity',
    'statute','class','severity_code'
]

existing_cols = [col for col in columns_to_drop if col in df.columns]
df.drop(columns=existing_cols, inplace=True)

print(f"✔ {len(existing_cols)} sütun kaldırıldı.")
print("Kaldırılanlar:", existing_cols)


✔ 16 sütun kaldırıldı.
Kaldırılanlar: ['X', 'Y', 'rin', 'primary_key', 'charge_rin', 'pin', 'ESRI_OID', 'x_coordinate', 'y_coordinate', 'arrest_officer', 'ofc_age_range', 'ofc_genderTranslation', 'ofc_RaceAndEthnicity', 'statute', 'class', 'severity_code']


In [4]:
# Eksik değerleri inceleme
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
display(missing.head(20))
print("\nVeri boyutu:", df.shape)


area_name                     5.667735
zipcode                       5.225107
arrest_type                   1.199309
arrest_translation            1.199309
ChargeGrouping                0.265576
arrestee_ethnicity            0.073771
arrestee_age_range            0.067448
arrestee_race                 0.040047
arrestee_RaceAndEthnicity     0.040047
arrestee_genderTranslation    0.040047
arrestee_sex                  0.040047
charge_count                  0.002108
arrest_time                   0.000000
arrest_dt                     0.000000
location                      0.000000
arrest_hour_of_day            0.000000
jurisdiction                  0.000000
municipality                  0.000000
zone                          0.000000
grid                          0.000000
dtype: float64


Veri boyutu: (47444, 24)


In [5]:
print("\n--- Aykırı Değer İşleme ---")

if "charge_count" in df.columns:
    Q1 = df['charge_count'].quantile(0.25)
    Q3 = df['charge_count'].quantile(0.75)
    IQR = Q3 - Q1
    upper = Q3 + 1.5*IQR

    anomalies = df[df['charge_count'] > upper].shape[0]

    df.loc[df['charge_count'] > upper, 'charge_count'] = np.nan
    print(f"✔ {anomalies} adet aykırı değer NaN yapıldı.")
else:
    print("⚠ 'charge_count' bulunamadı.")



--- Aykırı Değer İşleme ---
✔ 856 adet aykırı değer NaN yapıldı.


In [6]:
# Düşük kategorili verileri tespit etme
categorical_cols = df.select_dtypes(include=['object','category','bool']).columns

threshold = 0.001
print(f"\n--- Düşük Frekanslı Kategoriler (%{threshold*100}) ---")

for col in categorical_cols:
    vc = df[col].value_counts(normalize=True)
    low = vc[vc < threshold].index.tolist()
    if low:
        print(f"• {col}: {len(low)} düşük frekanslı kategori → {low[:3]}...")



--- Düşük Frekanslı Kategoriler (%0.1) ---
• arrest_type: 1 düşük frekanslı kategori → ['S']...
• arrest_translation: 1 düşük frekanslı kategori → ['Summoned/Cited: Ticket issued, or long form charges approved, not taken to jail']...
• arrest_dt: 20549 düşük frekanslı kategori → ['2023/02/11 20:00:00+00', '2024/05/09 16:10:00+00', '2023/06/02 10:14:00+00']...
• location: 2015 düşük frekanslı kategori → [' PRIEST DR / W SOUTHERN AVE  ', '4XX W BROADWAY RD    ', '1XXX E VISTA DEL CERRO DR     ']...
• municipality: 2 düşük frekanslı kategori → ['SR        ', 'CH        ']...
• district: 2 düşük frekanslı kategori → ['UI    ', 'DT    ']...
• zone: 2 düşük frekanslı kategori → ['UI    ', '1     ']...
• grid: 223 düşük frekanslı kategori → ['2311  ', '0606  ', '1317  ']...
• charge: 5889 düşük frekanslı kategori → ['PRESCRIPT DRUG-POSSESS/USE                                                                                                                                                       

In [7]:
print("\n--- İmputasyon Başladı (Uyarısız Güvenli Versiyon) ---")

# SAYISAL İMPUTASYON (MOD/MEDYAN)
num_cols = df.select_dtypes(include=np.number).columns

for col in num_cols:
    if df[col].isna().any():
        if df[col].nunique() < 10:
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            df[col] = df[col].fillna(df[col].median())

# KATEGORİK İMPUTASYON (DAĞILIM KORUMA)
cat_cols = df.select_dtypes(include=['object','category','bool']).columns

def impute_with_dist(col):
    nonnull = df[col].dropna()
    if nonnull.empty:
        return  
    probs = nonnull.value_counts(normalize=True)
    nan_mask = df[col].isna()
    nan_count = nan_mask.sum()
    if nan_count == 0:
        return
    random_values = np.random.choice(probs.index, size=nan_count, p=probs.values)
    df.loc[nan_mask, col] = random_values
    print(f"✔ '{col}' kategorik imputasyon ile dolduruldu.")

for col in cat_cols:
    if df[col].isna().any():
        impute_with_dist(col)

print("\n✔ Tüm imputasyonlar uyarısız olarak tamamlandı.")



--- İmputasyon Başladı (Uyarısız Güvenli Versiyon) ---
✔ 'arrest_type' kategorik imputasyon ile dolduruldu.
✔ 'arrest_translation' kategorik imputasyon ile dolduruldu.
✔ 'area_name' kategorik imputasyon ile dolduruldu.
✔ 'arrestee_age_range' kategorik imputasyon ile dolduruldu.
✔ 'ChargeGrouping' kategorik imputasyon ile dolduruldu.
✔ 'arrestee_sex' kategorik imputasyon ile dolduruldu.
✔ 'arrestee_race' kategorik imputasyon ile dolduruldu.
✔ 'arrestee_ethnicity' kategorik imputasyon ile dolduruldu.
✔ 'arrestee_genderTranslation' kategorik imputasyon ile dolduruldu.
✔ 'arrestee_RaceAndEthnicity' kategorik imputasyon ile dolduruldu.

✔ Tüm imputasyonlar uyarısız olarak tamamlandı.


In [8]:
print("\n--- Zaman Feature Engineering ---")

# Tutuklama tarihinden yıl, ay ve haftanın günü gibi yeni özellikler türetildi
if 'arrest_dt' in df.columns:
    df['arrest_dt'] = pd.to_datetime(df['arrest_dt'].astype(str).str.split().str[0], errors='coerce')
    df['arrest_year'] = df['arrest_dt'].dt.year
    df['arrest_month'] = df['arrest_dt'].dt.month
    df['arrest_day_of_week'] = df['arrest_dt'].dt.dayofweek

    # Yeni bir kategorik özellik oluşturuldu
    def get_time_cat(h):
        if pd.isna(h): return 'Bilinmiyor'
        if 0<=h<=6: return 'Gece'
        if 7<=h<=11: return 'Sabah'
        if 12<=h<=16: return 'Ogle'
        if 17<=h<=20: return 'Ikindi'
        return 'Aksam'

    df['time_category'] = df['arrest_hour_of_day'].apply(get_time_cat)

    df.drop(columns=[c for c in ['arrest_dt','arrest_time'] if c in df.columns], inplace=True)
    print("✔ Zaman özellikleri eklendi.")



--- Zaman Feature Engineering ---
✔ Zaman özellikleri eklendi.


In [9]:
print("\n--- Hedef Değişken ---")

df['is_onview_arrest'] = np.where(df['arrest_type']=='O', 1, 0)
df.drop(columns=[c for c in ['arrest_type','arrest_translation'] if c in df.columns],
        inplace=True)

print("✔ 'is_onview_arrest' oluşturuldu.")



--- Hedef Değişken ---
✔ 'is_onview_arrest' oluşturuldu.


In [10]:
df.to_csv("preprocessed_common.csv", index=False)
print("✔ Ortak preprocessing çıktısı kaydedildi.")


✔ Ortak preprocessing çıktısı kaydedildi.
